# Story of case A: phase-space pictures (v1, 23/10/2026)

Case A: $\kappa_b = \kappa_b^{(0)} = \pi/10.4$ (code convention; 96 MHz in the article's GKSL convention), where the satellite has a period-doubling bubble. $\varepsilon_d = 10\,g_2$ ($|\alpha|^2 = 5$), $g_2 = E_J\sin\varepsilon_p\,\varphi_a^2\varphi_b$.
One figure per range of $g_2$, one row per value of $g_2$, four panels per row:
- **(a) limit cycles**: harmonic-balance orbits in the rotating frame of the memory, $a_R = i\,a\,e^{i\tau}$ (solid: stable, dashed: unstable; the partner $a\to-a$, i.e. the orbit shifted by $T_a/2$, is lighter), and the last 20 periods of each trajectory (thin grey);
- **(b) Poincaré section** $a_R(\tau = 2\pi k)$: the section points of the orbits (markers; filled: stable) and of the trajectories (coloured by $\kappa_b t$), and the border of the chaotic set (black contour) when there is one;
- **(c) time trace** $|a_R(2\pi k)|$ against $\kappa_b t$ for each trajectory: transient chaos, landing;
- **(d) basins** on the slice $b(0) = 0$, $a_R(0)$ on a $32\times32$ grid (initial memory state, empty buffer, as in the Lyapunov maps): colour = end state after $N$ periods. "Chaotic" means not settled at the end of the run, so long transients are counted as chaotic.
  The local basin radius $\rho_{50}$ of the thin basins (kick size at which half of 8 random 4D kicks come back, in $x^\lambda$ units; $|\delta a| \lesssim \rho/(\sqrt2\varphi_a) = 6.4\,\rho$) is written in the panel.

Names: **cat**, **period-1 saddle** (middle branch of the S, between the two folds), **satellite** (period 1), **branch I** (period 2, born at the period doubling, 75.34 MHz), **branch II** (period 2, born at the reverse period doubling, 103.88 MHz),
**peripheric node** and **peripheric saddle** (the $\omega_a/2$ period-2 pair, born together at 99.97 MHz; in the rotating frame both wind once around the origin over $2T_a$).
Times are in $1/\kappa_b$ (code convention, $1/\kappa_b = 13.4\,T_a$). Time integrations: `story_A_trajectories.jl` (Julia, $\approx$ 20 min on 8 threads; skipped if the output exists and `RUN_JULIA = False`).

| range of $g_2$ (MHz) | step | $g_2$ shown |
|---|---|---|
| 0 – 40.65 | 1. only the cats | 30 |
| 40.65 – 75.34 | 2. satellite fold: satellite and period-1 saddle born | 60 |
| 75.34 – 76.80 | 3. period doubling of the satellite: branch I stable | 76.0 |
| 76.80 – 77.33 | 4. torus; chaotic saddle | 76.9, 77.3 |
| 77.33 – 97.83 | 5. chaotic attractor coexists with the cat | 85, 95 |
| 97.83 – 99.97 | 6. cat fold: the chaotic attractor alone | 99 |
| 99.97 – 100.24 | 7. peripheric node born inside the chaotic set: leak | 100.2 |
| 100.24 – 100.4 | 8. torus around the peripheric node | 100.3 |
| 100.4 – 101.15 | 9. chaotic attractor again | 100.9 |
| 101.2 – 103.88 | 10. peripheric node stable again: transient chaos | 102.4 |
| 103.88 – 106.71 | 11. satellite stable again (thin basin, branch II) | 105 |
| 106.71 – 111 | 12. peripheric node unstable: other destinations | 108 |
| 111 – 120 | 13. open: separate chaotic bands | 117 |

Step 4 at $\varepsilon_d = 10\,g_2$: Neimark–Sacker of branch I at 76.80 MHz (harmonic balance); the torus is the attractor up to 77.30–77.35 MHz (scan of 23/10: from the chaotic attractor of 80 MHz, every start lands on the torus at 77.30 MHz, after up to $6000\,T_a$, none within $4\times10^4\,T_a$ at 77.35 MHz).
`torus_breakdown.ipynb` found at $10.12\,g_2$: Neimark–Sacker 76.70, torus doubling 77.01, chaos 77.25 MHz; the torus doubling was not looked for here.

In [ ]:
import json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/pylib")
from catchaos.params import *                      # w_a, phi_a, phi_b, E_J, kappa_b (code convention), G0, MHz, g2_max
from catchaos.plotting import *                    # Plot_setting (figsize_car, Perso_palette_float, ...)
from catchaos.ats import HB, HB2, rotating_frame
from catchaos.hill import state, odd_content, monodromy
from catchaos.seeds import satellite_hb, seed_z
from catchaos.branches import PS, extended_g2, arclength_g2
warnings.filterwarnings("ignore", category=RuntimeWarning)

OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/"
FIG_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/torus_crisis/"

R = 10.0                             # eps_d / g2 (row of bubble_crisis_lifetime.jl)
N_H, N_H2 = 16, 32                   # harmonics: period 1, period 2 (N_H2 = 2 N_H, so the satellite embeds exactly)
G_C = 101.4                          # crisis of the bubble's attractor (satellite_start_bubble_edge, section 4)

## Harmonic balance and continuation
From `catchaos` (formerly copied from `Hill_period2_branch_II.ipynb`): `ats.HB`, `ats.HB2` (period-2 basis), `hb.newton`, `ats.rotating_frame`, `hill.state`, `hill.odd_content`, `hill.monodromy`,
`seeds.satellite_hb`, `seeds.seed_z`, and the pseudo-arclength in $p = g_2/(100\text{ MHz})$ `branches.arclength_g2`, `branches.extended_g2`, `branches.PS`.

## Limit cycles at every $g_2$ of the story
- **Period 1** (cat, period-1 saddle, satellite): one pseudo-arclength continuation of the S-shaped branch from the satellite at 60 MHz, down through the satellite fold, up along the saddle, through the cat fold and down along the cat to 20 MHz; the satellite above 60 MHz by natural continuation.
- **Period 2**: from the seeds of `period2_branches_eps_d10_23_10_2026.json` (branch I, branch II, peripheric node and saddle), natural continuation checked against the stored odd content.
For each orbit: stability (monodromy), the curve $a_R(\tau)$, its section points and those of its partner.

In [ ]:
import os, subprocess
from matplotlib.colors import ListedColormap
from scipy.ndimage import gaussian_filter
HERE = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/studies/torus_crisis/"
FIG_STORY = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/torus_crisis/story_A/"
os.makedirs(FIG_STORY, exist_ok=True)
B2 = json.load(open(OUT_DIR + "torus_crisis/period2_branches_eps_d10_23_10_2026.json"))
G_STORY = [30.0, 60.0, 76.0, 76.9, 77.3, 85.0, 95.0, 99.0, 100.2, 100.3, 100.9, 102.4, 105.0, 108.0, 117.0]
G_EXTRA = [97.5]                                    # only for the local basin of the cat near its fold

In [ ]:
# --- period 1: the S-shaped branch by arclength from the satellite at 60 MHz ---
hb = HB(N_H, r=R)
hb.set_s(60/g2_max); z60 = hb.solve(seed_z(satellite_hb(60/g2_max, R), N_H))
F_, dF_ = extended_g2(hb, R)
y0 = np.append(z60, 60/PS)
t_ = np.linalg.svd(dF_(y0))[2][-1]; t_ = t_ if t_[-1] < 0 else -t_
S_BR, end = arclength_g2(hb, y0, t_, R, ds_max=3e-3, g_lo=20.0, g_hi=140.0, n_max=20000, stop_odd=False)
gS = S_BR[:, -1]*PS
turn = np.where(np.diff(np.sign(np.diff(gS))) != 0)[0] + 1
print(f"S-branch: {len(S_BR)} points, {end}; folds at " + ", ".join(f"{gS[i]:.2f}" for i in turn) + " MHz")
g_fold_sat, g_fold_cat = gS[turn[0]], gS[turn[1]]
SEG = {"satellite": S_BR[:turn[0] + 1], "saddle1": S_BR[turn[0]:turn[1] + 1], "cat": S_BR[turn[1]:]}
SAT_UP, z = {}, z60.copy()
for g in np.arange(60.25, 120.01, 0.25):
    hb.set_s(g/g2_max); z = hb.newton(z); SAT_UP[round(g, 2)] = z.copy()

In [ ]:
def p1_at(name, g):
    if name == "satellite" and g > 60:
        g0 = max(k for k in SAT_UP if k <= g); z = SAT_UP[g0]
    else:
        br = SEG[name]; gb = br[:, -1]*PS
        if not (gb.min() <= g <= gb.max()):
            return None
        z = br[np.argmin(np.abs(gb - g)), :-1]
    hb.set_s(g/g2_max); return hb.newton(z)

def p2_at(name, g, dg=0.02):
    D = B2[name]; g_all, odd_all = np.array(D["g2_MHz"]), np.array(D["odd"])
    if not (g_all.min() <= g <= g_all.max()):
        return None
    gz = g_all[::D["z_every"]]; j = np.argmin(np.abs(gz - g)); z, g0 = np.array(D["z"][j]), gz[j]
    hb2 = HB2(N_H2, r=R)
    for gg in np.linspace(g0, g, max(1, int(np.ceil(abs(g - g0)/dg))) + 1)[1:]:
        hb2.set_s(gg/g2_max); z = hb2.newton(z)
    assert abs(odd_content(z, N_H2) - odd_all[np.argmin(np.abs(g_all - g))]) < 0.01, f"{name} jumped branch at {g} MHz"
    return z

In [ ]:
ORB_STYLE = {"cat": ("cat", Perso_palette_float[2]), "saddle1": ("period-1 saddle", "0.45"), "satellite": ("satellite", Perso_palette_float[5]),
             "branch_I": ("branch I", "tab:blue"), "branch_II": ("branch II", "crimson"),
             "omega_half_node": ("peripheric node", Perso_palette_float[3]), "omega_half_saddle": ("peripheric saddle", "darkorange")}
aR_of = lambda Y: 1j*(Y[0] + 1j*Y[2])/(np.sqrt(2)*phi_a)            # a_R on the section tau = 0 mod 2 pi

def orbit_record(name, z, g, period):
    N = N_H if period == 1 else N_H2
    hb_ = HB(N, r=R) if period == 1 else HB2(N, r=R); hb_.set_s(g/g2_max)
    mu = monodromy(hb_, z)
    tt = np.linspace(0, 2*np.pi*period, 400*period)
    a_R = rotating_frame(z, N, tt, period)[0]
    taus = 2*np.pi*np.arange(period)
    sec = [state(z, N, t, period) for t in taus]; par = [state(z, N, t + np.pi, period) for t in taus]
    return dict(name=name, g=g, z=z, period=period, N=N, mu=mu, stable=bool(np.abs(mu).max() < 1), a_R=a_R,
                sec=np.array(sec), sec_partner=np.array(par), sec_aR=np.array([aR_of(Y) for Y in sec]), par_aR=np.array([aR_of(Y) for Y in par]))

In [ ]:
ORBITS = {}
for g in G_STORY + G_EXTRA:
    recs = []
    for name in ("cat", "saddle1", "satellite"):
        z = p1_at(name, g)
        if z is not None:
            recs.append(orbit_record(name, z, g, 1))
    for name in ("branch_I", "branch_II", "omega_half_node", "omega_half_saddle"):
        z = p2_at(name, g)
        if z is not None:
            recs.append(orbit_record(name, z, g, 2))
    ORBITS[g] = {r["name"]: r for r in recs}
    desc = lambda r: ORB_STYLE[r["name"]][0] + (" (stable)" if r["stable"] else f" (max|mu| = {np.abs(r['mu']).max():.2f})")
    print(f"{g:6.1f} MHz: " + ", ".join(desc(r) for r in recs))

## Jobs for the time integrations
- **Trajectories** (per step): from $a(0) = i|\alpha|$, $b(0) = 0$ (the cat-start state of the maps), or from an orbit plus a random kick of given size, or from a state of the chaotic attractor taken at another $g_2$ (`pre`).
- **Basins**: $32\times32$ grid of $a_R(0) \in [-6.5, 6.5]^2$, $b(0) = 0$; $N = 3000\,T_a$ ($224/\kappa_b$), or $8000\,T_a$ where the chaos is transient with lifetimes of a few hundred $1/\kappa_b$.
- **Local basins** of the stable orbits whose basin is thin: kicks $\epsilon = 10^{-3.5}\dots1$, 8 random directions, up to $5000\,T_a$.

In [ ]:
RNG = np.random.default_rng(2310)
def lab_state(a, b=0.0):
    return [np.sqrt(2)*phi_a*a.real, np.sqrt(2)*phi_b*b.real, np.sqrt(2)*phi_a*a.imag, 2*np.sqrt(2)*phi_b*b.imag]
def kicked(g, name, eps):
    v = RNG.normal(size=4); return (ORBITS[g][name]["sec"][0] + eps*v/np.linalg.norm(v)).tolist()
A_CAT = 1j*np.sqrt(R/2)                                   # a(0) = i |alpha|
TRAJ = {   # g2: list of (label, u0, N_Ta, pre)
    30.0:  [("a(0) = i|α|", lab_state(A_CAT), 3000, None), ("a(0) = 3", lab_state(3 + 0j), 3000, None)],
    60.0:  [("a(0) = i|α|", lab_state(A_CAT), 3000, None), ("satellite + 1e-3", kicked(60.0, "satellite", 1e-3), 3000, None),
            ("period-1 saddle + 1e-3", kicked(60.0, "saddle1", 1e-3), 3000, None)],
    76.0:  [("satellite + 1e-4", kicked(76.0, "satellite", 1e-4), 5000, None), ("a(0) = i|α|", lab_state(A_CAT), 3000, None)],
    76.9:  [("satellite + 1e-4", kicked(76.9, "satellite", 1e-4), 20000, None)],
    77.3:  [("satellite + 1e-4", kicked(77.3, "satellite", 1e-4), 20000, None)] +
           [(f"from the chaotic attractor at 80 MHz (#{k})", kicked(77.3, "satellite", 1e-3), 20000, (80.0, 3000)) for k in range(3)],
    85.0:  [("satellite + 1e-3", kicked(85.0, "satellite", 1e-3), 20000, None), ("a(0) = i|α|", lab_state(A_CAT), 3000, None)],
    95.0:  [("satellite + 1e-3", kicked(95.0, "satellite", 1e-3), 20000, None), ("a(0) = i|α|", lab_state(A_CAT), 3000, None)],
    99.0:  [("satellite + 1e-3", kicked(99.0, "satellite", 1e-3), 20000, None), ("a(0) = i|α|", lab_state(A_CAT), 20000, None)],
    100.2: [(f"satellite + 1e-3 (#{k})", kicked(100.2, "satellite", 1e-3), 100000, None) for k in range(4)],
    100.3: [("peripheric node + 1e-4", kicked(100.3, "omega_half_node", 1e-4), 20000, None), ("satellite + 1e-3", kicked(100.3, "satellite", 1e-3), 50000, None)],
    100.9: [("peripheric node + 1e-5", kicked(100.9, "omega_half_node", 1e-5), 20000, None), ("satellite + 1e-3", kicked(100.9, "satellite", 1e-3), 20000, None)],
    102.4: [(f"satellite + 1e-3 (#{k})", kicked(102.4, "satellite", 1e-3), 30000, None) for k in range(3)],
    105.0: [("satellite + 0.002", kicked(105.0, "satellite", 0.002), 20000, None), ("satellite + 0.05", kicked(105.0, "satellite", 0.05), 20000, None),
            ("a(0) = i|α|", lab_state(A_CAT), 20000, None)],
    108.0: [(f"satellite + 0.05 (#{k})", kicked(108.0, "satellite", 0.05), 20000, None) for k in range(2)],
    117.0: [(f"satellite + 0.05 (#{k})", kicked(117.0, "satellite", 0.05), 30000, None) for k in range(2)] + [("a(0) = i|α|", lab_state(A_CAT), 30000, None)],
}

In [ ]:
N_BASIN = {g: 3000 for g in G_STORY}
N_BASIN.update({102.4: 8000, 105.0: 8000, 108.0: 8000, 117.0: 8000})
KICKS = [(60.0, "cat"), (60.0, "satellite"), (85.0, "cat"), (95.0, "cat"), (97.5, "cat"), (76.0, "branch_I"),
         (100.2, "omega_half_node"), (102.4, "omega_half_node"), (105.0, "omega_half_node"), (105.0, "satellite")]
EPS_K = np.logspace(-3.5, 0, 15).tolist()
NG = 32
AR_GRID = np.linspace(-6.5, 6.5, NG)

In [ ]:
jobs = []
for g, lst in TRAJ.items():
    for k, (lab, u0, N, pre) in enumerate(lst):
        J = dict(kind="traj", id=f"traj_{g}_{k}", g2_MHz=g, ratio=R, u0=list(map(float, u0)), N_Ta=N, dense=[[0, 3], [N - 20, 20]])
        if pre is not None:
            J["pre"] = list(pre)
        jobs.append(J)
for g in G_STORY:
    for i, im in enumerate(AR_GRID):
        jobs.append(dict(kind="basin", id=f"basin_{g}_{i}", g2_MHz=g, ratio=R, row=i, N_Ta=N_BASIN[g],
                         u0s=[lab_state(-1j*(re + 1j*im)) for re in AR_GRID]))
for g, name in KICKS:
    o = ORBITS[g][name]
    jobs.append(dict(kind="kick", id=f"kick_{g}_{name}", g2_MHz=g, ratio=R, ref=[list(map(float, s)) for s in np.r_[o["sec"], o["sec_partner"]]],
                     eps=EPS_K, n_dir=8, N_max_Ta=5000))
import hashlib
for J in jobs:
    J["hash"] = hashlib.md5(json.dumps(J, sort_keys=True).encode()).hexdigest()
print(f"{len(jobs)} jobs: " + ", ".join(f"{k} {sum(j['kind'] == k for j in jobs)}" for k in ("traj", "basin", "kick")))

In [ ]:
# results are cached by job: only new or changed jobs (different hash) are run; RUN_JULIA = True reruns everything
OUT_J = OUT_DIR + "torus_crisis/story_A_23_10_2026.json"
RUN_JULIA = False
cache = {} if (RUN_JULIA or not os.path.exists(OUT_J)) else {r["id"]: r for r in json.load(open(OUT_J))["runs"]}
todo = [J for J in jobs if J["id"] not in cache or cache[J["id"]].get("hash") != J["hash"]]
if todo:
    IN_J, TMP_J = OUT_DIR + "torus_crisis/story_A_input_23_10_2026.json", OUT_DIR + "torus_crisis/story_A_new_23_10_2026.json"
    json.dump({"jobs": todo}, open(IN_J, "w"))
    t0 = time.time()
    run = subprocess.run(["julia", "-t", "8", "--project=/Users/tmalas/Desktop/Dev/Simulations_local/Code_Julia", HERE + "story_A_trajectories.jl", IN_J, TMP_J],
                         capture_output=True, text=True)
    print(f"ran {len(todo)} jobs ({time.time() - t0:.0f} s)", run.stdout[-300:], run.stderr[-1500:])
    new = json.load(open(TMP_J)); os.remove(TMP_J)
    cache.update({r["id"]: r for r in new["runs"]})
    ids = {J["id"] for J in jobs}
    json.dump({"T_a_over_kappa_inv": new["T_a_over_kappa_inv"], "runs": [r for k, r in cache.items() if k in ids]}, open(OUT_J, "w"))
else:
    print("all jobs cached")
KT_A = 2*np.pi*kappa_b/w_a
RUN = cache

## Classification of the end states, local basins, border of the chaotic set
- **End state** of a basin run (last 64 section states): periodic with period $p \le 8$ (to $10^{-5}$), then matched to a stable orbit of the list (within $10^{-3}$, orbit or partner); otherwise "other periodic orbit".
  Not periodic but all 64 points within 0.35 of the section points of an orbit: that orbit if it is stable (still converging), "torus" if it is unstable (a quasi-periodic set around it); otherwise "chaotic (not settled)".
- **Border of the chaotic set**: outline of the region of the section visited by the chaotic part of a trajectory (iterates farther than 0.05, in $x^\lambda$ units, from the set of its last 500 iterates, i.e. before it lands), from a smoothed histogram of $a_R(2\pi k)$.
- **Fate of a trajectory** (legend of panel c): end state from its last 64 iterates (same classification), and the landing time = last iterate farther than 0.05 from its final set.

In [ ]:
CLASSES = ["cat", "satellite", "branch I", "torus", "peripheric node", "other periodic", "chaotic (not settled)"]
CLASS_COL = [Perso_palette_float[2], Perso_palette_float[5], "tab:blue", "plum", Perso_palette_float[3], "0.7", (0.55, 0.05, 0.05)]
NAME_CLASS = {"cat": 0, "satellite": 1, "branch_I": 2, "omega_half_node": 4}

def classify(last, g):
    L = np.array(last)
    for p in range(1, 9):
        if np.abs(L[p:] - L[:-p]).max() < 1e-5:
            for name, o in ORBITS[g].items():
                if o["stable"] and name in NAME_CLASS:
                    S = np.r_[o["sec"], o["sec_partner"]]
                    if np.linalg.norm(L[-1][None] - S, axis=1).min() < 1e-3:
                        return NAME_CLASS[name], p
            return 5, p
    for name, o in ORBITS[g].items():
        if name in NAME_CLASS:
            S = np.r_[o["sec"], o["sec_partner"]]
            if np.linalg.norm(L[:, None] - S[None], axis=2).min(axis=1).max() < 0.35:
                return (NAME_CLASS[name] if o["stable"] else 3), 0     # still converging onto a stable orbit, or a torus around an unstable one
    return 6, 0

In [ ]:
BASIN, PERIODS = {}, {}
for g in G_STORY:
    M_b = np.zeros((NG, NG), int); per = {}
    for i in range(NG):
        for j, last in enumerate(RUN[f"basin_{g}_{i}"]["last"]):
            c, p = classify(last, g); M_b[i, j] = c
            if c == 5:
                per[p] = per.get(p, 0) + 1
    BASIN[g], PERIODS[g] = M_b, per
    frac = {CLASSES[c]: np.mean(M_b == c) for c in range(len(CLASSES)) if np.any(M_b == c)}
    print(f"{g:6.1f} MHz (N = {N_BASIN[g]} T_a): " + ", ".join(f"{k} {100*v:.0f} %" for k, v in frac.items()) + (f"; other periodic orbits: periods {per}" if per else ""))

In [ ]:
RHO50 = {}
for g, name in KICKS:
    fr = np.array(RUN[f"kick_{g}_{name}"]["frac_back"]); e = np.array(EPS_K)
    i = np.where(fr < 0.5)[0]
    RHO50[(g, name)] = np.exp(np.interp(0.5, [fr[i[0]], fr[i[0] - 1]], [np.log(e[i[0]]), np.log(e[i[0] - 1])])) if len(i) and i[0] > 0 else (e[0] if len(i) else np.inf)
    print(f"local basin of the {ORB_STYLE[name][0]} at {g} MHz: rho_50 = {RHO50[(g, name)]:.3g}" + (" (> 1: every kick comes back)" if np.isinf(RHO50[(g, name)]) else ""))

In [ ]:
def traj(g, k):
    r = RUN[f"traj_{g}_{k}"]; S = np.array(r["section"]); t = np.array(r["index"])*KT_A
    L = S[-500:]; d = np.array([np.linalg.norm(L - Y, axis=1).min() for Y in S]); far = d > 0.05
    c, p = classify(S[-64:], g)
    fate = (CLASSES[c] if c != 5 else f"period-{p} orbit") if c != 6 else "still chaotic"
    t_land = t[np.where(far)[0][-1]] if far.any() else 0.0
    fate += "" if c == 6 else (f" after {t_land:.0f}" + r"$/\kappa_b$" if t_land > 1 else "")
    return dict(t=t, S=S, aR=np.array([aR_of(Y) for Y in S]), dense=r["dense"], far=far, fate=fate)

EXT = 7.0
def chaos_border(g, ks):
    pts = []
    for k in ks:
        T = traj(g, k); pts.append(T["aR"][T["far"]])
    pts = np.concatenate(pts)
    if len(pts) < 500:
        return None
    H, xe, ye = np.histogram2d(pts.real, pts.imag, bins=140, range=[[-EXT, EXT], [-EXT, EXT]])
    return (xe[:-1] + xe[1:])/2, (ye[:-1] + ye[1:])/2, gaussian_filter((H > 0).astype(float), 1.5).T

## Figure helper

In [ ]:
TRAJ_CMAPS = ["viridis", "plasma", "cividis", "magma"]
TRAJ_COLS = ["k", "tab:purple", "tab:green", "tab:brown"]

def draw_orbits(ax, g, curves=True, markers=True, ms=5):
    for name, o in ORBITS[g].items():
        lab, col = ORB_STYLE[name]
        if curves:
            ls = "-" if o["stable"] else "--"
            ax.plot(o["a_R"].real, o["a_R"].imag, ls, color=col, lw=0.9, label=lab)
            ax.plot(-o["a_R"].real, -o["a_R"].imag, ls, color=col, lw=0.9, alpha=0.35)
        if markers:
            for P, alpha in ((o["sec_aR"], 1.0), (o["par_aR"], 0.5)):
                ax.plot(P.real, P.imag, "o" if o["period"] == 1 else "s", ms=ms, mfc=col if o["stable"] else "none", mec=col, mew=1.0, alpha=alpha, zorder=6)

def square(ax, lim=EXT):
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.set_xlabel(r"Re $a_R$"); ax.set_ylabel(r"Im $a_R$")

In [ ]:
def story_row(axes, g, border_from=None, zoom=None, note=""):
    lst = TRAJ[g]
    # (a) limit cycles and the last 20 periods of each trajectory
    ax = axes[0]
    for k in range(len(lst)):
        T = traj(g, k); key = max(T["dense"], key=int); D = np.array(T["dense"][key])
        tt = np.linspace(0, 2*np.pi*20, len(D))
        aR = 1j*(D[:, 0] + 1j*D[:, 2])/(np.sqrt(2)*phi_a)*np.exp(1j*tt)
        ax.plot(aR.real, aR.imag, color="0.6", lw=0.3, zorder=1)
    draw_orbits(ax, g, markers=False); square(ax)
    ax.set_title(rf"$g_2/2\pi = {g:g}$ MHz: limit cycles (HB) {note}", fontsize=6)
    ax.legend(fontsize=4.5, loc="lower left")
    # (b) Poincaré section
    ax = axes[1]
    for k, (lab, *_rest) in enumerate(lst):
        T = traj(g, k)
        ax.scatter(T["aR"].real, T["aR"].imag, c=T["t"], cmap=TRAJ_CMAPS[k % 4], s=0.6, lw=0, zorder=2)
    B = chaos_border(g, border_from) if border_from is not None else None
    if B is not None:
        ax.contour(*B, levels=[0.05], colors="k", linewidths=0.6, zorder=3)
    draw_orbits(ax, g, curves=False)
    square(ax)
    if zoom is not None:
        ax.set_xlim(zoom[0], zoom[1]); ax.set_ylim(zoom[2], zoom[3])
    ax.set_title(r"section $a_R(2\pi k)$; colour: $\kappa_b t$" + ("; black: border of the chaotic set" if B is not None else ""), fontsize=6)
    # (c) time traces
    ax = axes[2]
    for k, (lab, *_rest) in reversed(list(enumerate(lst))):
        T = traj(g, k)
        ax.plot(T["t"], np.abs(T["aR"]), ".", ms=0.6 if k else 1.2, color=TRAJ_COLS[k % 4], label=f"{lab} → {T['fate']}")
    ax.set_xlabel(r"$\kappa_b t$"); ax.set_ylabel(r"$|a_R(2\pi k)|$"); ax.legend(fontsize=4.5, markerscale=8, loc="upper right")
    # (d) basins on the slice b(0) = 0
    ax = axes[3]
    ax.pcolormesh(AR_GRID, AR_GRID, BASIN[g], cmap=ListedColormap(CLASS_COL), vmin=-0.5, vmax=len(CLASSES) - 0.5, shading="nearest", rasterized=True)
    if B is not None:
        ax.contour(*B, levels=[0.05], colors="k", linewidths=0.6)
    draw_orbits(ax, g, curves=False, ms=4); square(ax, 6.5)
    txt = "; ".join(f"{ORB_STYLE[n][0]}: " + (r"$\rho_{50} > 1$" if np.isinf(r) else rf"$\rho_{{50}} = {r:.2g}$") for (gg, n), r in RHO50.items() if gg == g)
    fr = ", ".join(f"{CLASSES[c] if c != 5 else 'other periodic ' + str(sorted(PERIODS[g]))} {100*np.mean(BASIN[g] == c):.0f} %" for c in range(len(CLASSES)) if np.any(BASIN[g] == c))
    ax.set_title(rf"basins, $b(0) = 0$, $N = {N_BASIN[g]}\,T_a$: " + fr + (f"\n{txt}" if txt else ""), fontsize=5.5)

In [ ]:
def story_figure(rows, fname, title):
    fig, axes = plt.subplots(len(rows), 4, figsize=(4.4*figsize_car[0], 1.15*len(rows)*figsize_car[1]), squeeze=False)
    for ax_row, kw in zip(axes, rows):
        story_row(ax_row, **kw)
    present = sorted({c for kw in rows for c in np.unique(BASIN[kw["g"]])})
    handles = [mpl.patches.Patch(color=CLASS_COL[c]) for c in present] + \
              [mpl.lines.Line2D([], [], color="k", ls="-", lw=0.9), mpl.lines.Line2D([], [], color="k", ls="--", lw=0.9),
               mpl.lines.Line2D([], [], ls="", marker="o", mfc="k", mec="k"), mpl.lines.Line2D([], [], ls="", marker="s", mfc="none", mec="k")]
    labels = [CLASSES[c] for c in present] + ["stable orbit", "unstable orbit", "section point, period 1 (filled: stable)", "section point, period 2 (open: unstable)"]
    fig.legend(handles, labels, fontsize=5, ncol=6, loc="lower center", bbox_to_anchor=(0.5, -0.02 - 0.03/len(rows)), frameon=False)
    fig.suptitle(title, fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_STORY + fname, bbox_inches="tight", dpi=250)
    plt.show()

# 1. $g_2 < 40.65$ MHz: only the cats
The two cats ($\pm$) are the only limit cycles. Every start ends on one of them.

In [ ]:
story_figure([{'g': 30.0}], 'story_A_1_cats_23_10_2026.pdf', '1. $g_2 < 40.65$ MHz: only the cats')

# 2. 40.65–75.34 MHz: the satellite fold
At 40.65 MHz the satellite and the period-1 saddle are born together (saddle-node on the period-1 branch). Cat and satellite are both stable; the saddle sits between them.

In [ ]:
story_figure([{'g': 60.0}], 'story_A_2_satellite_fold_23_10_2026.pdf', '2. 40.65–75.34 MHz: the satellite fold')

# 3. 75.34–76.80 MHz: period doubling of the satellite
The satellite period-doubles (supercritical): it becomes unstable and the stable period-2 orbit branch I is born (two section points), with a local basin $\rho_{50} = 0.045$. The cat is unaffected. On the slice $b(0) = 0$: branch I 70 %, cat 6 %, and 24 % still not settled after $224/\kappa_b$, i.e. long chaotic transients already exist here.

In [ ]:
story_figure([{'g': 76.0, 'zoom': (-1.2, 1.2, 2.2, 4.6)}], 'story_A_3_period_doubling_23_10_2026.pdf', '3. 75.34–76.80 MHz: period doubling of the satellite')

# 4. 76.80–77.33 MHz: torus; chaotic saddle
Branch I has a Neimark–Sacker point at 76.80 MHz: a torus (two invariant circles on the section, one per point of branch I) replaces it as the attractor. A chaotic set already exists, but it is not an attractor (chaotic saddle): just below the end of the torus (77.3 MHz), starts from the chaotic attractor of 80 MHz wander on it for $10^2$–$10^3/\kappa_b$, then land on the torus.

In [ ]:
story_figure([{'g': 76.9, 'zoom': (-1.2, 1.2, 2.2, 4.6)}, {'g': 77.3, 'border_from': [1, 2, 3]}], 'story_A_4_torus_23_10_2026.pdf', '4. 76.80–77.33 MHz: torus; chaotic saddle')

# 5. 77.33–97.83 MHz: chaotic attractor and cat
The torus is gone and the chaotic set is an attractor. It coexists with the stable cat. On the slice $b(0) = 0$ the cat's basin is small (2 % at 85 MHz, none of the grid points at 95 MHz), and its local radius shrinks towards the fold: $\rho_{50} = 0.15$ (60 MHz), 0.14 (85), 0.11 (95), 0.04 (97.5 MHz). At 95 MHz the cat-start state $a(0) = i|\alpha|$ is already outside it.

In [ ]:
story_figure([{'g': 85.0, 'border_from': [0]}, {'g': 95.0, 'border_from': [0]}], 'story_A_5_chaos_and_cat_23_10_2026.pdf', '5. 77.33–97.83 MHz: chaotic attractor and cat')

# 6. 97.83–99.97 MHz: the chaotic attractor alone
The cat and the period-1 saddle have merged at the cat fold (97.83 MHz). The chaotic attractor is almost the only attractor: 98 % of the slice is chaotic, 2 % ends on a period-3 orbit (not identified).

In [ ]:
story_figure([{'g': 99.0, 'border_from': [0, 1]}], 'story_A_6_chaos_alone_23_10_2026.pdf', '6. 97.83–99.97 MHz: the chaotic attractor alone')

# 7. 99.97–100.24 MHz: the peripheric pair is born inside the chaotic set
Saddle-node at 99.97 MHz: the peripheric node and saddle (period 2) appear in the region visited by the chaotic set. The node is stable ($\rho_{50} = 0.037$), its basin is a hole, and the chaos leaks into it ($\tau \sim 10^4$–$10^5/\kappa_b$: 3 of the 4 runs land within $7500/\kappa_b$). On the slice, 5 % ends on the node within $224/\kappa_b$, 1 % on the period-3 orbit.

In [ ]:
story_figure([{'g': 100.2, 'border_from': [0, 1, 2, 3]}], 'story_A_7_peripheric_birth_23_10_2026.pdf', '7. 99.97–100.24 MHz: the peripheric pair is born inside the chaotic set')

# 8. 100.24–100.4 MHz: torus around the peripheric node
The peripheric node has a Neimark–Sacker point at 100.24 MHz. A start next to it settles on an invariant circle around it; starts from the chaotic set can be captured there.

In [ ]:
story_figure([{'g': 100.3, 'border_from': [1]}], 'story_A_8_peripheric_torus_23_10_2026.pdf', '8. 100.24–100.4 MHz: torus around the peripheric node')

# 9. 100.4–101.15 MHz: chaotic attractor again
The torus around the node is gone; the node is unstable and embedded in the chaotic attractor (a start next to it leaves for the chaos).

In [ ]:
story_figure([{'g': 100.9, 'border_from': [0, 1]}], 'story_A_9_chaos_again_23_10_2026.pdf', '9. 100.4–101.15 MHz: chaotic attractor again')

# 10. 101.2–103.88 MHz: the peripheric node stable again, transient chaos
At 101.20 MHz the node restabilises inside the chaotic set: its basin opens a hole, and the chaos becomes transient for good (here $\tau \approx 475/\kappa_b$). Every trajectory ends on the node or its partner (66 % of the slice within $600/\kappa_b$; the rest still in the transient). The local basin grows with $g_2$: $\rho_{50} = 0.083$ here (0.007 at 101.22 MHz, `crisis_collision.ipynb`).

In [ ]:
story_figure([{'g': 102.4, 'border_from': [0, 1, 2]}], 'story_A_10_transient_chaos_23_10_2026.pdf', '10. 101.2–103.88 MHz: the peripheric node stable again, transient chaos')

# 11. 103.88–106.71 MHz: the satellite stable again
Reverse period doubling at 103.88 MHz (subcritical): the satellite is stable again, inside a thin basin bounded by branch II (period-2 saddle): $\rho_{50} = 0.0056$, so no grid point of the slice reaches it. A kick of 0.002 comes back; a kick of 0.05 goes through transient chaos to the peripheric node, which takes 99 % of the slice ($\rho_{50} = 0.087$). (The satellite and one point of the node's partner have the same $|a_R|$ on the section, 2.47, hence the overlapping lines in panel c.)

In [ ]:
story_figure([{'g': 105.0, 'border_from': [1]}], 'story_A_11_satellite_again_23_10_2026.pdf', '11. 103.88–106.71 MHz: the satellite stable again')

# 12. 106.71–111 MHz: the peripheric node unstable
The node has a Neimark–Sacker point at 106.71 MHz. Transient chaos now ends elsewhere: at 108 MHz 96 % of the slice ends on a **large period-1 orbit that winds once around the origin** ($|a_R| \approx 4.6$, grey circle in panel a; not in the harmonic-balance list yet), and 1 % on a period-5 orbit.

In [ ]:
story_figure([{'g': 108.0, 'border_from': [0, 1]}], 'story_A_12_other_destinations_23_10_2026.pdf', '12. 106.71–111 MHz: the peripheric node unstable')

# 13. 111–120 MHz: open
Separate chaotic bands (111–114, 116–120 MHz) whose origin is not explained yet. At 117 MHz: 84 % of the slice still chaotic after $600/\kappa_b$ (two runs still chaotic after $2200/\kappa_b$), 16 % on a period-5 orbit.

In [ ]:
story_figure([{'g': 117.0, 'border_from': [0, 1, 2]}], 'story_A_13_open_23_10_2026.pdf', '13. 111–120 MHz: open')